# How to aggregate a time series

**Goal:** turn a long time series into a few typical periods you can hand to a model. This is
the core recipe; the other guides cover specific variations (make it smaller, change the
grouping, preserve peaks, map results back). For a guided walk-through see the
[tutorial](../tutorials/quickstart.ipynb); for the theory, see
[How aggregation works](../explanation/how-aggregation-works/00_overview.ipynb).

## 1. Load your data

tsam expects a `DataFrame` with a **datetime index** and one column per attribute.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data
UNITS = {"GHI": "W/m²", "T": "°C", "Wind": "m/s", "Load": "MW"}
data.head()

## 2. Aggregate

One call does it. Two parameters carry the decision:

* **`n_clusters`** — how many typical periods to keep (more = more accurate, less reduction).
* **`period_duration`** — the length of one period: `"1D"` (daily), `"1W"` (weekly), or a
  number of hours (`24`, `168`).

If your index is irregular or its step cannot be inferred, also pass
**`temporal_resolution`** (e.g. `"15min"`).

In [ ]:
result = tsam.aggregate(data, n_clusters=8, period_duration="1D")
print("reduced", len(data), "hours to", result.n_clusters, "typical days")

## 3. Read the outputs

Three pieces are what a downstream model consumes:

* **`cluster_representatives`** — the typical period profiles.
* **`cluster_counts`** — how many real periods each one stands for (its weight).
* **`accuracy`** — RMSE / MAE per column, to judge whether the reduction is acceptable.

In [ ]:
print("counts:", result.cluster_counts)
print("\nper-column RMSE:")
print(result.accuracy.rmse.round(3).to_string())
result.cluster_representatives.head()

Check it held up — original vs. reconstructed over one week:

In [ ]:
result.plot.compare(
    columns=["Load"],
    time_slice=slice("2010-01-11", "2010-01-17"),
    color="source",
    units=UNITS,
    title="One week: original vs. reconstructed Load",
)

## 4. Common adjustments

Each is its own short guide — reach for them once the basic call is in place:

* **Weight columns by importance** — pass `weights={"Load": 2.0}` so an attribute has more
  pull on the clustering. (Covered in [Optimization workflow](optimization_workflow.ipynb).)
* **Make it smaller** — also reduce *within* each period with
  [Segmentation](segmentation.ipynb), or let tsam search the size/accuracy trade-off in
  [How small can you go?](tuning.ipynb).
* **Change how periods are grouped** — pick a method in
  [Clustering methods](clustering_methods.ipynb).
* **Choose what each typical period preserves** — pick a
  [representation](representations.ipynb), or force the peak day to survive with
  [Extreme periods](extreme_periods.ipynb).
* **Use the result** — read the typical-day ↔ original links and map model results back in
  [Working with typical periods](working_with_typical_periods.ipynb).

In [ ]:
# example: give Load twice the pull on the clustering
weighted = tsam.aggregate(
    data, n_clusters=8, period_duration="1D", weights={"Load": 2.0}
)
print(
    "Load RMSE — unweighted:",
    round(result.accuracy.rmse["Load"], 3),
    " weighted:",
    round(weighted.accuracy.rmse["Load"], 3),
)